<a href="https://colab.research.google.com/github/pranjalbainsla/ml-notes/blob/master/makemore_exercises_pt1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

E01: train a trigram language model, i.e. take two characters as an input to predict the 3rd one. Feel free to use either counting or a neural net. Evaluate the loss; Did it improve over a bigram model?

In [1]:
words = open("./names.txt").read().splitlines()

In [3]:
import torch
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}


/Users/pranjalbainsla/Desktop/ml/ml-notes/.venv/lib/python3.12/site-packages/torch/nn/modules/transformer.py:20: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  device: torch.device = torch.device(torch._C._get_default_device()),  # torch.device('cpu'),


In [4]:
N = torch.zeros((729, 27), dtype=torch.int32)
for w in words:
  chs = ['.', '.'] + list(w) + ['.']
  for i in range(len(chs)-2):
    ix1a = stoi[chs[i]]
    ix1b = stoi[chs[i+1]]


    ix1 = 27 * ix1a + ix1b 
    ix2 = stoi[chs[i+2]]

    N[ix1, ix2] += 1

In [ ]:
import matplotlib.pyplot as plt
row_labels = [itos[i // 27] + itos[i % 27] for i in range(N.shape[0])]

plt.figure(figsize=(20, 120), dpi=200)
plt.imshow(N, cmap="Blues", aspect="auto", interpolation="nearest")
plt.xticks(range(27), [itos[i] for i in range(27)])
plt.yticks(range(N.shape[0]), row_labels, fontsize=3)
plt.tight_layout()
plt.show()

In [5]:
N[0]

tensor([   0, 4410, 1306, 1542, 1690, 1531,  417,  669,  874,  591, 2422, 2963,
        1572, 2538, 1146,  394,  515,   92, 1639, 2055, 1308,   78,  376,  307,
         134,  535,  929], dtype=torch.int32)

In [6]:
P = (N+1).float() # P is 729 * 27

In [7]:
torch.sum(P, 1, keepdim=True).shape

torch.Size([729, 1])

In [8]:
P = P/torch.sum(P, 1, keepdim=True)

In [9]:
P[0]

tensor([3.1192e-05, 1.3759e-01, 4.0767e-02, 4.8129e-02, 5.2745e-02, 4.7785e-02,
        1.3038e-02, 2.0898e-02, 2.7293e-02, 1.8465e-02, 7.5577e-02, 9.2452e-02,
        4.9064e-02, 7.9195e-02, 3.5777e-02, 1.2321e-02, 1.6095e-02, 2.9008e-03,
        5.1154e-02, 6.4130e-02, 4.0830e-02, 2.4641e-03, 1.1759e-02, 9.6070e-03,
        4.2109e-03, 1.6719e-02, 2.9008e-02])

In [10]:
P[0].sum()

tensor(1.)

In [11]:
g = torch.Generator().manual_seed(2147483647)

for i in range(5):

  out = []
  prev1 = 0   # '.'
  prev2 = 0   # '.'

  while True:
      row = 27 * prev1 + prev2
      p = P[row]

      next_char = torch.multinomial(p, 1).item()

      if next_char == 0:
          break

      out.append(itos[next_char])

      prev1, prev2 = prev2, next_char

  print(''.join(out))

man
tonatharijachavorree
asevyjni
lane
aley


In [12]:
log_likelihood = 0.0
n = 0

for w in words:
#for w in ["andrejq"]:
  chs = ['.', '.'] + list(w) + ['.']
  for i in range(len(chs)-2):
    ix1a = stoi[chs[i]]
    ix1b = stoi[chs[i+1]]

    ix1 = 27 * ix1a + ix1b
    ix2 = stoi[chs[i+2]]

    prob = P[ix1, ix2]
    logprob = torch.log(prob)
    log_likelihood += logprob
    n += 1
    #print(f'{ch1}{ch2}: {prob:.4f} {logprob:.4f}')

print(f'{log_likelihood=}')
nll = -log_likelihood
print(f'{nll=}')
print(f'{nll/n}')

log_likelihood=tensor(-504653.)
nll=tensor(504653.)
2.2119739055633545


Did the loss improve over a bigram model? Yeah (2.45 to 2.21)